In [ ]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

In [ ]:
# Check the GPU status
!nvidia-smi

# Libraries

In [ ]:
%%capture
!uv pip uninstall torchao torchaudio torchvision -y
!uv pip install \
    "transformers==4.53.3" \
    "peft==0.17.1" \
    "trl" \
    "accelerate" \
    "bitsandbytes" \
    "wandb"

In [ ]:
from datetime import datetime
from transformers import AutoModelForQuestionAnswering, AutoTokenizer
from peft import PeftModel

# Configurations

In [ ]:
# Run configuration
SRC_LANG = 'en'
TGT_LANG = 'tr'

# Model configuration
# MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-en-1K-LoRA-Merged-v260623145250'
# LORA_ID = 'alxxtexxr/XLM-R-Base-wikipedia-vi-1K-LoRA-v260622154525'
# LORA_CKPT_DIR = 'checkpoint-60'
# ---
# MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-en-5K-LoRA-Merged-v260711104723'
# LORA_ID = 'alxxtexxr/XLM-R-Base-wikipedia-vi-5K-LoRA-v260719113226'
# LORA_CKPT_DIR = 'checkpoint-120'
# ---
MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-en-15K-s42-LoRA-mrg-v260925013329'
LORA_ID = 'alxxtexxr/XLM-R-Base-wikipedia-tr-3.5K-s42-LoRA-v260929054247'
LORA_CKPT_STEP = 800

# Set up the hub merged model ID
if SRC_LANG in MODEL_ID:
    model_id_pt_0, model_id_pt_1 = MODEL_ID.split(SRC_LANG)
elif TGT_LANG in LORA_ID:
    model_id_pt_0, model_id_pt_1 = LORA_ID.split(TGT_LANG)
else:
    raise ValueError("Model and LoRA IDs do not match the specified source language and target language.")
model_id_pt_1_0 = model_id_pt_1.split("K-")[-1].split("-LoRA")[0]
hub_merged_model_id = f"{model_id_pt_0}{TGT_LANG}-{model_id_pt_1_0}-LoRA-a2m-v{datetime.now().strftime("%y%m%d%H%M%S")}"
print(f"Hub merged model ID: {hub_merged_model_id}")

# Model

In [ ]:
# Load the base model
base_model = AutoModelForQuestionAnswering.from_pretrained(MODEL_ID, device_map='auto')
print("device:", base_model.device)

In [ ]:
# Sanity check
for i in range(11):
    print(f"Base model layer-{i} attention value weight norm:", base_model.roberta.encoder.layer[i].attention.self.value.weight.norm().item())
print()
print("Base model qa_outputs weight norm:", base_model.qa_outputs.weight.norm().item())
print("Base model qa_outputs bias norm:", base_model.qa_outputs.bias.norm().item())

In [ ]:
# Load and merge the LoRA adapter into the base model
lora_model = PeftModel.from_pretrained(base_model, subfolder=f"checkpoint-{LORA_CKPT_STEP}", model_id=LORA_ID)
merged_model = lora_model.merge_and_unload()

In [ ]:
# Sanity check
for i in range(11):
    print(f"Merged model layer-{i} attention value weight norm:", merged_model.roberta.encoder.layer[i].attention.self.value.weight.norm().item())
print()
print("Merged model qa_outputs weight norm:", merged_model.qa_outputs.weight.norm().item())
print("Merged model qa_outputs bias norm:", merged_model.qa_outputs.bias.norm().item())

In [ ]:
# Upload the merged model to Hugging Face
merged_model.push_to_hub(hub_merged_model_id)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.push_to_hub(hub_merged_model_id)

print(f"Merged model uploaded to: https://huggingface.co/{hub_merged_model_id}")